# 04 – Pilot λ_roi dan GAN (CP4, D2, FR-06)

GAN kondisional-kelas bergaya StyleGAN2 dengan D_roi (decisions.yaml D8). λ_roi dipilih dari **FID pada pasien validasi** di dalam lipatan latih (rep 0, lipatan 0); kriteria ditetapkan di konfigurasi sebelum eksperimen. Citra uji tidak dipakai. Tetapkan `gan.total_kimg` lebih dulu dari `benchmark_gan.json` (CP0).

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.


In [ ]:
bg = json.loads(P(cfg, 'cp0', 'benchmark_gan.json').read_text())
print('total_kimg di konfigurasi:', cfg['gan']['total_kimg'], '| estimasi jam 20 GAN:', round(bg['estimasi_jam_20_gan'], 1), '(dibagi jumlah GPU)')
print('Bila terlalu lama/pendek, ubah gan.total_kimg di configs/base.yaml SEBELUM menjalankan sel berikut.')

## 1. Pilot (dapat dilanjutkan; job DONE dilewati)

In [ ]:
rc = run_module('src.pilot_lambda', ['--config', 'configs/s6.yaml']); print('kode keluar:', rc)

## 2. Hasil pilot dan pemeriksaan visual

In [ ]:
sel = json.loads(P(cfg, 'cp4', 'lambda_selection.json').read_text())
print('lambda_roi terpilih:', sel['chosen'], '|', sel['source'])
display(pd.DataFrame(sel['table']))
for r in sel['table']:
    g = Path(r['dir']) / 'grid_val' / 'samples_malignant.png'
    print('lambda =', r['lambda_roi'])
    display(Image(filename=str(g)))

Periksa grid. Bila tidak ada mode collapse yang jelas: `python -m src.approve gan-visual CP4 --by NAMA`.

## 3. Pemeriksaan checkpoint CP4 dan `manifest.json` (sel terakhir)

In [ ]:
from src.checkpoint import finalize_cp4
report = finalize_cp4(cfg)
assert report['checkpoint'] == 'CP4'